# Level 4: Real-world decorators

## 4.1 Timer: how long does a function take?

In [ ]:
import functools
import time

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

slow_sum(5_000_000)

## 4.2 Logger: record calls and results

In [ ]:
def log_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        arg_parts = [repr(a) for a in args] + [f"{k}={v!r}" for k, v in kwargs.items()]
        print(f"-> {func.__name__}({', '.join(arg_parts)})")
        result = func(*args, **kwargs)
        print(f"<- {func.__name__} returned {result!r}")
        return result
    return wrapper

@log_calls
def make_full_name(first, last, title=""):
    return f"{title} {first} {last}".strip()

make_full_name("Anant", "Patel", title="Mr.")

## 4.3 Call counter: store state on the wrapper

Functions are objects, so we can attach attributes to them.

In [ ]:
def count_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

@count_calls
def ping():
    return "pong"

for _ in range(5):
    ping()
print("ping was called", ping.calls, "times")

## 4.4 Simple authorisation check

A decorator can decide **not** to call the function at all.

In [ ]:
current_user = {"name": "anant", "role": "viewer"}

def admin_only(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        if current_user["role"] != "admin":
            raise PermissionError(f"{current_user['name']} cannot call {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@admin_only
def delete_database():
    return "Database deleted!"

try:
    delete_database()
except PermissionError as e:
    print("Blocked:", e)

current_user["role"] = "admin"
print(delete_database())

### Try it (Level 4)
Write a decorator `uppercase` that converts the **return value** of a function to uppercase.

```python
@uppercase
def greet(name):
    return f"hello {name}"

greet("anant")   # 'HELLO ANANT'
```

In [ ]:
# your code here

### Solution
Try it yourself first, then compare with the cell below.

In [ ]:
# Solution
import functools

def uppercase(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper()
    return wrapper

@uppercase
def greet(name):
    return f"hello {name}"
print(greet("anant"))